# Qwen2.5-3B-Instruct — FinancialPhraseBank Sentiment Pipeline

**Flow**

1. Load **FinancialPhraseBank** (75% agreement) — stratified **60/20/20** (train/val/test) split.
2. **Stage 1 – Baseline**: frozen base model on FPB **test set** (20%).
3. **Stage 2 – QLoRA + Random Search** (small space): train on 60%, EarlyStopping on eval_loss, select by val **macro-F1**.
4. **Final**: retrain on train+val (80%) with best params → evaluate on **test set**.
5. All outputs → `genai_grp_project/qwen2.5_3b_sentiment/`.


In [1]:
%%capture
!pip install -U unsloth transformers==4.56.2 trl==0.22.2 peft bitsandbytes accelerate datasets scikit-learn pandas python-dotenv requests

In [2]:
import subprocess, torch

try:
    r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(r.stdout or r.stderr)
except FileNotFoundError:
    print("nvidia-smi not found")

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)
DTYPE = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
)
print(f"Device: {DEVICE} | DType: {DTYPE}")

Wed Apr  8 09:00:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   70C    P0             32W /   70W |    7365MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import os, re, gc, io, random
from pathlib import Path
import numpy as np, pandas as pd, requests
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    matthews_corrcoef,
    classification_report,
)

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

for p in [Path(os.getcwd()), *Path(os.getcwd()).parents]:
    if (p / ".env").exists():
        load_dotenv(p / ".env", override=False)
        print(f"Loaded .env from {p}")
        break

CKPT_BASE = (
    Path(os.getenv("CHECKPOINT_DIR", "genai_grp_project")) / "qwen2.5_3b_sentiment"
)
CKPT_BASE.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"
MAX_SEQ_LEN = 512
LABELS = ["negative", "neutral", "positive"]
LABEL_SET = set(LABELS)
RANDOM_SEARCH_TRIALS = int(os.getenv("RANDOM_SEARCH_TRIALS", 4))

print(f"CKPT_BASE : {CKPT_BASE}\nTrials    : {RANDOM_SEARCH_TRIALS}")

CKPT_BASE : genai_grp_project/qwen2.5_3b_sentiment
Trials    : 4


In [4]:
def make_prompt(s):
    return (
        "You are a financial sentiment classifier.\n"
        "Reply with exactly one word: negative, neutral, or positive.\n\n"
        f"Sentence: {s}\nLabel:"
    )


def make_chat_train_text(tok, sentence, label):
    return tok.apply_chat_template(
        [
            {
                "role": "system",
                "content": "Financial sentiment classifier. One word reply: negative, neutral, or positive.",
            },
            {"role": "user", "content": f"Classify: {sentence}"},
            {"role": "assistant", "content": label},
        ],
        tokenize=False,
        add_generation_prompt=False,
    )


def parse_label(text):
    m = re.search(r"(negative|neutral|positive)", text.strip().lower())
    return m.group(1) if m else "neutral"


@torch.inference_mode()
def batched_generate_labels(
    model, tokenizer, sentences, batch_size=32, max_new_tokens=3
):
    preds = []
    tokenizer.padding_side = "left"
    for i in range(0, len(sentences), batch_size):
        prompts = [make_prompt(s) for s in sentences[i : i + batch_size]]
        enc = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LEN,
        ).to(model.device)
        out = model.generate(
            **enc,
            do_sample=False,
            temperature=None,
            top_p=None,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )
        gen = out[:, enc["input_ids"].shape[1] :]
        preds.extend(
            parse_label(t)
            for t in tokenizer.batch_decode(gen, skip_special_tokens=True)
        )
    return preds


def evaluate_predictions(true_labels, pred_labels):
    true_labels = [x.strip().lower() for x in true_labels]
    pred_labels = [x if x in LABEL_SET else "neutral" for x in pred_labels]
    return {
        "macro_f1": f1_score(
            true_labels, pred_labels, labels=LABELS, average="macro", zero_division=0
        ),
        "weighted_f1": f1_score(
            true_labels, pred_labels, labels=LABELS, average="weighted", zero_division=0
        ),
        "accuracy": accuracy_score(true_labels, pred_labels),
        "mcc": matthews_corrcoef(true_labels, pred_labels),
        "f1_negative": f1_score(
            true_labels,
            pred_labels,
            labels=["negative"],
            average="macro",
            zero_division=0,
        ),
        "f1_neutral": f1_score(
            true_labels,
            pred_labels,
            labels=["neutral"],
            average="macro",
            zero_division=0,
        ),
        "f1_positive": f1_score(
            true_labels,
            pred_labels,
            labels=["positive"],
            average="macro",
            zero_division=0,
        ),
        "report": classification_report(
            true_labels, pred_labels, labels=LABELS, zero_division=0
        ),
    }


def print_metrics(title, m):
    print("=" * 70, title, "=" * 70, m["report"], sep="\n")
    for k in (
        "macro_f1",
        "weighted_f1",
        "accuracy",
        "mcc",
        "f1_negative",
        "f1_neutral",
        "f1_positive",
    ):
        print(f"  {k}: {m[k]:.4f}")


def free_gpu(*objs):
    for o in objs:
        del o
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [5]:
FPB_URL = (
    "https://raw.githubusercontent.com/maxwellsarpong/"
    "NLP-financial-text-processing-dataset/master/Sentences_75Agree.txt"
)

resp = requests.get(FPB_URL, timeout=60)
resp.raise_for_status()
fpb_df = pd.read_csv(
    io.StringIO(resp.text),
    sep="@",
    header=None,
    names=["sentence", "label_text"],
    engine="python",
    encoding="latin-1",
)
fpb_df["sentence"] = fpb_df["sentence"].str.strip()
fpb_df["label_text"] = fpb_df["label_text"].str.strip().str.lower()
fpb_df = fpb_df[fpb_df["label_text"].isin(LABEL_SET)].reset_index(drop=True)
print(f"FPB: {len(fpb_df)} samples")
print(fpb_df["label_text"].value_counts())

train_val_df, test_df = train_test_split(
    fpb_df, test_size=0.20, random_state=SEED, stratify=fpb_df["label_text"]
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.25, random_state=SEED, stratify=train_val_df["label_text"]
)
print(f"\nTrain {len(train_df)} | Val {len(val_df)} | Test {len(test_df)}")

FPB: 3453 samples
label_text
neutral     2146
positive     887
negative     420
Name: count, dtype: int64

Train 2071 | Val 691 | Test 691


## Stage 1 — Baseline (frozen base model on FPB test set)


In [6]:
from unsloth import FastLanguageModel

base_model, base_tok = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)
base_model.eval()

base_preds = batched_generate_labels(
    base_model, base_tok, test_df["sentence"].tolist(), batch_size=32
)
base_metrics = evaluate_predictions(test_df["label_text"].tolist(), base_preds)
print_metrics("Qwen2.5-3B — Zero-shot baseline on FPB test", base_metrics)

# Explicitly delete from global scope before clearing GPU cache.
# free_gpu() only deletes its local 'o' reference — the globals must be
# removed here so the GC can actually reclaim the VRAM.
del base_model, base_tok
free_gpu()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-04-08 09:00:53.330937: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775638853.358028    2386 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775638853.366807    2386 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775638853.390214    2386 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775638853.390246    2386 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775638853.390249    2386 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Qwen2.5-3B — Zero-shot baseline on FPB test
              precision    recall  f1-score   support

    negative       0.85      0.93      0.89        84
     neutral       0.93      0.78      0.85       429
    positive       0.65      0.88      0.74       178

    accuracy                           0.82       691
   macro avg       0.81      0.86      0.83       691
we

## Stage 2 — QLoRA + Random Search (val macro-F1 for model selection)


In [7]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq, EarlyStoppingCallback
from unsloth.chat_templates import train_on_responses_only

SEARCH_SPACE = {
    "lora_r": [8, 16],
    "lora_alpha": [16, 32],
    "lora_dropout": [0.0, 0.05],
    "learning_rate": [1e-4, 2e-4],
    "batch_size": [8],
    "grad_accum": [2, 4],
    "epochs": [2, 3],
}
rng = random.Random(SEED)
sample_params = lambda: {k: rng.choice(v) for k, v in SEARCH_SPACE.items()}


def build_hf_ds(tok, df_rows):
    return Dataset.from_dict(
        {
            "text": [
                make_chat_train_text(tok, s, y)
                for s, y in zip(df_rows["sentence"], df_rows["label_text"])
            ]
        }
    )


best_val_f1 = -1.0
best_params = None
best_trial = -1
trial_log = []

for trial in range(RANDOM_SEARCH_TRIALS):
    params = sample_params()
    print(
        f"\n{'=' * 60}\nTrial {trial + 1}/{RANDOM_SEARCH_TRIALS}: {params}\n{'=' * 60}"
    )

    model, tok = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=params["lora_r"],
        lora_alpha=params["lora_alpha"],
        lora_dropout=params["lora_dropout"],
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )

    trial_ckpt = CKPT_BASE / f"trial_{trial}"
    trainer = SFTTrainer(
        model=model,
        tokenizer=tok,
        train_dataset=build_hf_ds(tok, train_df),
        eval_dataset=build_hf_ds(tok, val_df),
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tok),
        packing=False,
        args=SFTConfig(
            output_dir=str(trial_ckpt),
            num_train_epochs=params["epochs"],
            per_device_train_batch_size=params["batch_size"],
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=params["grad_accum"],
            learning_rate=params["learning_rate"],
            lr_scheduler_type="cosine",
            warmup_ratio=0.05,
            weight_decay=0.01,
            optim="adamw_8bit",
            logging_steps=20,
            eval_strategy="steps",
            eval_steps=50,
            eval_accumulation_steps=4,
            prediction_loss_only=True,
            save_strategy="steps",
            save_steps=50,
            load_best_model_at_end=True,
            metric_for_best_model="loss",
            greater_is_better=False,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            dataloader_num_workers=4,
            seed=SEED,
            report_to="none",
        ),
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=2, early_stopping_threshold=0.001
            )
        ],
    )
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )
    trainer.train()
    del trainer
    gc.collect()

    FastLanguageModel.for_inference(model)
    model.eval()
    try:
        val_preds = batched_generate_labels(
            model, tok, val_df["sentence"].tolist(), batch_size=16
        )
        val_metrics = evaluate_predictions(val_df["label_text"].tolist(), val_preds)
        val_f1 = val_metrics["macro_f1"]
    except torch.cuda.OutOfMemoryError:
        print(f"OOM trial {trial + 1} – skip")
        del model, tok
        free_gpu()
        continue

    print(f"Trial {trial + 1} | val macro-F1: {val_f1:.4f}")
    trial_log.append({"trial": trial + 1, "val_macro_f1": val_f1, "params": params})

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_params = params
        best_trial = trial + 1
        model.save_pretrained(str(CKPT_BASE / "best_adapter"))
        tok.save_pretrained(str(CKPT_BASE / "best_adapter"))
        print(
            f"  ↑ New best saved (trial {best_trial}, val macro-F1={best_val_f1:.4f})"
        )

    # Explicitly delete from global scope — free_gpu() alone can't reclaim
    # VRAM because it only removes its own local references, not these globals.
    del model, tok
    free_gpu()

if best_params is None:
    raise RuntimeError("All trials failed.")
print(
    f"\nBest trial {best_trial} | val macro-F1 {best_val_f1:.4f} | params {best_params}"
)


Trial 1/4: {'lora_r': 8, 'lora_alpha': 16, 'lora_dropout': 0.05, 'learning_rate': 0.0001, 'batch_size': 8, 'grad_accum': 2, 'epochs': 2}
==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.4 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/2071 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/691 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/2071 [00:00<?, ? examples/s]

...

Filter (num_proc=8):   0%|          | 0/691 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,071 | Num Epochs = 3 | Total steps = 390
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 14,966,784 of 3,100,905,472 (0.48% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
50,0.115400,0.068613
100,0.082600,0.073537
150,0.040100,0.144979


Trial 4 | val macro-F1: 0.9108

Best trial 2 | val macro-F1 0.9185 | params {'lora_r': 8, 'lora_alpha': 32, 'lora_dropout': 0.0, 'learning_rate': 0.0001, 'batch_size': 8, 'grad_accum': 2, 'epochs': 2}


In [8]:
# ─── Retrain on train+val with best params ────────────────────────────────────
FINAL_CKPT = CKPT_BASE / "final_best"

final_model, final_tok = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
final_model = FastLanguageModel.get_peft_model(
    final_model,
    r=best_params["lora_r"],
    lora_alpha=best_params["lora_alpha"],
    lora_dropout=best_params["lora_dropout"],
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

final_trainer = SFTTrainer(
    model=final_model,
    tokenizer=final_tok,
    train_dataset=build_hf_ds(final_tok, train_val_df),
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    data_collator=DataCollatorForSeq2Seq(tokenizer=final_tok),
    packing=False,
    args=SFTConfig(
        output_dir=str(FINAL_CKPT),
        num_train_epochs=best_params["epochs"],
        per_device_train_batch_size=best_params["batch_size"],
        per_device_eval_batch_size=best_params["batch_size"] * 2,
        gradient_accumulation_steps=best_params["grad_accum"],
        learning_rate=best_params["learning_rate"],
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        weight_decay=0.01,
        optim="adamw_8bit",
        logging_steps=20,
        save_strategy="epoch",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        dataloader_num_workers=4,
        seed=SEED,
        report_to="none",
    ),
)
final_trainer = train_on_responses_only(
    final_trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
final_trainer.train()
final_model.save_pretrained(str(FINAL_CKPT / "adapter"))
final_tok.save_pretrained(str(FINAL_CKPT / "adapter"))
print(f"Adapter saved → {FINAL_CKPT / 'adapter'}")
del final_trainer
gc.collect()
FastLanguageModel.for_inference(final_model)
final_model.eval()
print("Model ready for Stage 3 evaluation.")

==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/2762 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/2762 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/2762 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,762 | Num Epochs = 2 | Total steps = 346
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 14,966,784 of 3,100,905,472 (0.48% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
20,0.147000
40,0.153900
60,0.097200
80,0.049800
100,0.076600
120,0.045500
140,0.033900
160,0.054100
180,0.043400
200,0.018000


Adapter saved → genai_grp_project/qwen2.5_3b_sentiment/final_best/adapter
Model ready for Stage 3 evaluation.


## Stage 3 — Final Evaluation on Test Set (best hyperparams from Stage 2)


In [9]:
# Evaluate the best model (trained with Stage 2 best hyperparams) on the held-out test set
test_preds = batched_generate_labels(
    final_model, final_tok, test_df["sentence"].tolist(), batch_size=16
)
final_metrics = evaluate_predictions(test_df["label_text"].tolist(), test_preds)
print_metrics(
    f"Qwen2.5-3B QLoRA (Stage 2 best trial {best_trial}) — FPB TEST SET", final_metrics
)

Qwen2.5-3B QLoRA (Stage 2 best trial 2) — FPB TEST SET
              precision    recall  f1-score   support

    negative       0.96      0.95      0.96        84
     neutral       0.98      0.94      0.96       429
    positive       0.88      0.98      0.93       178

    accuracy                           0.95       691
   macro avg       0.94      0.96      0.95       691
weighted avg       0.95      0.95      0.95       691

  macro_f1: 0.9485
  weighted_f1: 0.9512
  accuracy: 0.9508
  mcc: 0.9107
  f1_negative: 0.9581
  f1_neutral: 0.9595
  f1_positive: 0.9280


In [10]:
# Save merged model for direct inference in agents/sentiment.py
# Merged 16-bit model loadable with AutoModelForCausalLM.from_pretrained()
MERGED_DIR = CKPT_BASE / "merged_model_16bit"
print(f"Saving merged model -> {MERGED_DIR}")
final_model.save_pretrained_merged(
    str(MERGED_DIR), final_tok, save_method="merged_16bit"
)
print("Merged model saved.")

import json as _json

(CKPT_BASE / "inference_config.json").write_text(
    _json.dumps(
        {
            "model_type": "causal_lm",
            "model_name_or_path": str(MERGED_DIR),
            "base_model": MODEL_NAME,
            "adapter_path": str(FINAL_CKPT / "adapter"),
            "labels": LABELS,
            "val_macro_f1": round(best_val_f1, 4),
            "best_trial": best_trial,
            "best_params": best_params,
        },
        indent=2,
    ),
    encoding="utf-8",
)
print(f"inference_config.json -> {CKPT_BASE}")

Saving merged model -> genai_grp_project/qwen2.5_3b_sentiment/merged_model_16bit


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:32<00:00, 16.35s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:41<00:00, 20.55s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/genai_grp_project/qwen2.5_3b_sentiment/merged_model_16bit`
Merged model saved.
inference_config.json -> genai_grp_project/qwen2.5_3b_sentiment


In [11]:
rows = [
    {
        "stage": "Baseline (zero-shot)",
        **{k: v for k, v in base_metrics.items() if isinstance(v, float)},
    },
    {
        "stage": f"QLoRA best (trial {best_trial})",
        **{k: v for k, v in final_metrics.items() if isinstance(v, float)},
    },
]
cols = [
    "stage",
    "macro_f1",
    "weighted_f1",
    "accuracy",
    "mcc",
    "f1_negative",
    "f1_neutral",
    "f1_positive",
]
display(
    pd.DataFrame(rows)[cols]
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)
print("\nTrial log:", *[f"  {r}" for r in trial_log], sep="\n")

,stage,macro_f1,weighted_f1,accuracy,mcc,f1_negative,f1_neutral,f1_positive
0,QLoRA best (trial 2),0.948536,0.951228,0.950796,0.910651,0.958084,0.959524,0.92800
1,Baseline (zero-shot),0.825748,0.824950,0.820550,0.698996,0.886364,0.846252,0.74463



Trial log:
  {'trial': 1, 'val_macro_f1': 0.8790857029858903, 'params': {'lora_r': 8, 'lora_alpha': 16, 'lora_dropout': 0.05, 'learning_rate': 0.0001, 'batch_size': 8, 'grad_accum': 2, 'epochs': 2}}
  {'trial': 2, 'val_macro_f1': 0.9185206444393255, 'params': {'lora_r': 8, 'lora_alpha': 32, 'lora_dropout': 0.0, 'learning_rate': 0.0001, 'batch_size': 8, 'grad_accum': 2, 'epochs': 2}}
  {'trial': 3, 'val_macro_f1': 0.8575610247935511, 'params': {'lora_r': 8, 'lora_alpha': 16, 'lora_dropout': 0.05, 'learning_rate': 0.0001, 'batch_size': 8, 'grad_accum': 4, 'epochs': 2}}
  {'trial': 4, 'val_macro_f1': 0.9107709001421173, 'params': {'lora_r': 8, 'lora_alpha': 32, 'lora_dropout': 0.05, 'learning_rate': 0.0002, 'batch_size': 8, 'grad_accum': 2, 'epochs': 3}}


In [12]:
for v in ["final_model", "final_tok", "base_model", "base_tok"]:
    if v in globals():
        del globals()[v]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Done.")

Done.
